## Physics-Motivated Geometric Representation for Propagation-State Prediction in 3D UAV Communication Environments

# Part 2: BaseLine2 Geometric Representation Extraction (Ladan et. al.)

Geometric feature extraction for UAV-to-ground wireless channel modeling.

This module reads UAV and ground-receiver coordinates together with building
geometry stored in DXF files. It extracts geometric features describing the
UAV-receiver link and the surrounding building environment.

The processing pipeline is:

1. Read UAV and ground-receiver coordinates from CSV files.
2. Extract building footprints and heights from a DXF file.
3. Compute building-related features within a configurable section around
   each UAV transmitter.
4. Compute geometric features for every UAV-RX pair.
5. Merge the transmitter-level and link-level features.
6. Save the resulting feature dataset as a CSV file.
7. Optionally visualize the first UAV-RX link and the building geometry.




# 1. Imports

In [ ]:
import numpy as np
import pandas as pd
from shapely.geometry import Point
import ezdxf
import matplotlib.pyplot as plt
from collections import Counter
import os
from matplotlib.patches import Polygon
import geopandas as gpd
from shapely.geometry import Polygon
from shapely.geometry import box

# 2. Experiment Configuration
This section defines the main parameters used for the experiments.

In [ ]:
BASE_DIR = Path(".")
REPRESENTATION_DIR = BASE_DIR / "Inputs_DIR"
OUTPUT_DIR = BASE_DIR / "REPRESENTATION_DIR"

# Dataset-specific input/output configuration.
Process each configured simulation scenario.

Each city entry specifies the UAV coordinates, receiver coordinates,
building geometry, and output file associated with that scenario.

In [ ]:
# Simulation scenarios.
# Each scenario defines the UAV/transmitter coordinates, ground-receiver
# coordinates, building DXF geometry, and destination file for the
# generated baseline features.
cities = {
    1: {
        "name": "Seoul",
        "uav_file" : REPRESENTATION_DIR / "Seoul_South Korea_UAV_Cartesian.csv",
        "gnd_file" : REPRESENTATION_DIR / "Seoul_South Korea_GND_Cartesian.csv",
        "dxf_file" : REPRESENTATION_DIR / "Seoul_Buildings.dxf",
        "output_file" : OUTPUT_DIR / "Seoul_Ladan_features.csv",        
    },
    2: {
        "name": "Boston",
        "uav_file" : REPRESENTATION_DIR / "BOSTON_UAV_Cartesian.csv",
        "gnd_file" : REPRESENTATION_DIR / "BOSTON_GND_Cartesian.csv",
        "dxf_file" : REPRESENTATION_DIR / "Boston_Building.dxf",
        "output_file" : OUTPUT_DIR / "Boston_Ladan_features.csv",        
    },
    3: {
        "name": "Pisa_East",
        "uav_file" : REPRESENTATION_DIR / "Pisa_East_UAV_Cartesian.csv",
        "gnd_file" : REPRESENTATION_DIR / "Pisa_East_GND_Cartesian.csv",
        "dxf_file" : REPRESENTATION_DIR / "Pisa_East_Building.dxf",
        "output_file" : OUTPUT_DIR / "Pisa_East_Ladan_features.csv",        
    },
    4: {
        "name": "Pisa_West",
        "uav_file": REPRESENTATION_DIR / "Pisa_West_UAV_Cartesian.csv",
        "gnd_file": REPRESENTATION_DIR / "Pisa_West_GND_Cartesian.csv",
        "dxf_file": REPRESENTATION_DIR / "Pisa_West_Building.dxf",
        "output_file" : OUTPUT_DIR / "Pisa_West_Ladan_features.csv",        
    },
    5: {
        "name": "London",
        "uav_file": REPRESENTATION_DIR / "London_UAV_Cartesian.csv",
        "gnd_file": REPRESENTATION_DIR / "London_GND_Cartesian.csv",
        "dxf_file": REPRESENTATION_DIR / "London_Building.dxf",
        "output_file" : OUTPUT_DIR / "London_Ladan_features.csv" ,        
    },
    6: {
        "name": "Manhattan",
        "uav_file" : REPRESENTATION_DIR /  "Manhattan_UAV_Cartesian.csv",
        "gnd_file" : REPRESENTATION_DIR /  "Manhattan_GND_Cartesian.csv",
        "dxf_file" : REPRESENTATION_DIR /  "Manhattan_Building.dxf",
        "output_file"  : OUTPUT_DIR /  "Manhattan_Ladan_features.csv" ,        
    },
    7: {
        "name": "Mobile_Seoul1",
        "uav_file" : REPRESENTATION_DIR /  "Mobile_Seoul_South Korea_UAV_TR_Cartesian.csv",
        "gnd_file" : REPRESENTATION_DIR /  "Mobile_Seoul_South Korea_GND_TR_Cartesian.csv",
        "dxf_file" : REPRESENTATION_DIR /  "Seoul_Buildings.dxf",
        "output_file" : OUTPUT_DIR /  "Mobile_Seoul_South Korea_TR_Ladan_features.csv" ,        
    },   
}
   

# Experiment Functions
The following functions implement data loading, and feature extraction.

In [ ]:

def read_tx_rx(uav_file, gnd_file):
    """
    Read UAV transmitter and ground-receiver coordinates from CSV files.

    The CSV files must contain the columns:
        Cartesian_X, Cartesian_Y, Cartesian_Z

    Parameters
    ----------
    uav_file : str or pathlib.Path
        Path to the CSV file containing UAV transmitter coordinates.
    gnd_file : str or pathlib.Path
        Path to the CSV file containing ground-receiver coordinates.

    Returns
    -------
    uav_df : pandas.DataFrame
        Complete UAV input table.
    gnd_df : pandas.DataFrame
        Complete ground-receiver input table.
    uav_coords : numpy.ndarray
        UAV coordinates with shape (N_uav, 3).
    gnd_coords : numpy.ndarray
        Ground-receiver coordinates with shape (N_rx, 3).
    """
    uav_df = pd.read_csv(uav_file)
    gnd_df = pd.read_csv(gnd_file)    
    uav_coords = uav_df[['Cartesian_X', 'Cartesian_Y', 'Cartesian_Z']].values
    gnd_coords = gnd_df[['Cartesian_X', 'Cartesian_Y', 'Cartesian_Z']].values    
    return uav_df, gnd_df, uav_coords, gnd_coords


def plot_dxf_with_first_tx_rx(dxf_file, uav_coords, gnd_coords):
    """
    Plot building geometry together with the first UAV transmitter and
    ground receiver.

    Building geometry is read from 3DFACE entities in the DXF file.
    The first UAV and first ground receiver are displayed together with
    the direct link between them.

    This function is intended primarily for debugging and visual inspection
    of the input geometry and transmitter-receiver locations.

    Parameters
    ----------
    dxf_file : str or pathlib.Path
        Path to the DXF file containing building geometry.
    uav_coords : numpy.ndarray
        UAV coordinates with shape (N_uav, 3).
    gnd_coords : numpy.ndarray
        Ground-receiver coordinates with shape (N_rx, 3).

    Returns
    -------
    None
        Displays a matplotlib figure.
    """
    
    doc = ezdxf.readfile(dxf_file)
    msp = doc.modelspace()
    types = Counter(entity.dxftype() for entity in msp)
    print(types)   
    fig, ax = plt.subplots(figsize=(10, 10))
    # Project DXF 3DFACE entities onto the XY plane to visualize
    # building footprints.
    for face in msp.query("3DFACE"):
        pts = []       
        for i in range(4):
            v = face.dxf.get(f"vtx{i}")
            pts.append((v.x, v.y))        
        unique_pts = []
        for p in pts:
            if p not in unique_pts:
                unique_pts.append(p)
        poly = Polygon(unique_pts,closed=True,edgecolor="gray",facecolor="lightgray",linewidth=0.2)           
        ax.add_patch(poly)    
    tx = uav_coords[0]
    ax.scatter(tx[0], tx[1],c="red",s=120,marker="^",label="TX (UAV)")
    ax.text(tx[0],tx[1],"TX",color="red",fontsize=12)    
    rx = gnd_coords[0]
    ax.scatter(rx[0], rx[1],c="blue",s=80,marker="o",label="RX (Ground)")
    ax.text(rx[0],rx[1],"RX",color="blue",fontsize=12)    
    ax.plot([tx[0], rx[0]],[tx[1], rx[1]],"g--",linewidth=1)
    ax.set_aspect("equal")
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.set_title("DXF Buildings with First TX and RX")
    ax.legend()
    # Automatically adjust the plot limits to include both the building
    # geometry and the UAV/RX locations.
    ax.autoscale()
    plt.show()


In [ ]:


def extract_tx_features(buildings_gdf, tx_coords, section_size=100):
    """
    Extract building-environment features around each UAV transmitter.

    For each UAV transmitter, a square section centered at the UAV's
    XY position is created. Buildings intersecting this section are
    identified and used to calculate descriptive statistics of the
    surrounding built environment.

    The extracted features include the number of buildings, building-height
    statistics, building density, the number of buildings above the average
    height, and the total number of building faces and vertices.

    Parameters
    ----------
    buildings_gdf : geopandas.GeoDataFrame
        Building geometries and associated attributes. The GeoDataFrame
        must contain the columns ``geometry``, ``height``, ``num_faces``,
        and ``num_vertices``.
    tx_coords : numpy.ndarray
        UAV transmitter coordinates with shape (N_uav, 3).
    section_size : float, optional
        Side length of the square analysis section in meters.
        Default is 100 m.

    Returns
    -------
    pandas.DataFrame
        One row per UAV transmitter containing the extracted
        building-environment features.
    """
    features = []
    # Convert section size to the half-width used to construct the square.
    half = section_size / 2
    for tx_id, (x, y, z) in enumerate(tx_coords):
        # Define a square analysis region centered on the UAV in the XY plane.
        section = box(x-half,y-half,x+half,y+half)
        nearby = buildings_gdf[buildings_gdf.intersects(section)]
        if nearby.empty:
            features.append({
                "TX": tx_id + 1,
                "Num_buildings": 0,
                "Maximum_height": 0,
                "Minimum_height": 0,
                "Average_height": 0,
                "Median_Height": 0,
                "Density_Ratio": 0,
                "Num_building_above_avg": 0,
                "Num_faces": 0,
                "Num_vertices": 0
            })
            continue
        heights = nearby["height"].values
        avg_height = heights.mean()
        # Calculate the fraction of the section area occupied by building
        # geometries. Intersections are used so that buildings crossing the
        # section boundary contribute only their portion inside the section.
        density = (nearby.geometry.intersection(section).area.sum()/ section.area)
        features.append({
            "TX": tx_id + 1,
            "Num_buildings": len(nearby),
            "Maximum_height": heights.max(),
            "Minimum_height": heights.min(),
            "Average_height": avg_height,
            "Median_Height": np.median(heights),
            "Density_Ratio": density,
            "Num_building_above_avg": np.sum(heights > avg_height),
            "Num_faces": nearby["num_faces"].sum(),
            "Num_vertices": nearby["num_vertices"].sum()
        })
    return pd.DataFrame(features)

def extract_all_features_Baseline2(output_file,uav_coords,gnd_coords):
    """
    Generate geometric features for every UAV-RX pair.

    Each UAV transmitter is paired with every ground receiver. For each
    pair, the Euclidean 3D distance and the Cartesian coordinates of both
    endpoints are stored.

    Parameters
    ----------
    output_file : str or pathlib.Path or None
        Optional path for saving the generated feature table as CSV.
        If None, the table is returned without being saved.
    uav_coords : numpy.ndarray
        UAV transmitter coordinates with shape (N_uav, 3).
    gnd_coords : numpy.ndarray
        Ground-receiver coordinates with shape (N_rx, 3).

    Returns
    -------
    pandas.DataFrame
        One row for every UAV-RX combination. The resulting table contains
        transmitter and receiver identifiers, their Cartesian coordinates,
        and the 3D Euclidean distance between them.
    """
    features = []
    for tx_id, (tx_x, tx_y, tx_z) in enumerate(uav_coords):
        for rx_id, (rx_x, rx_y, rx_z) in enumerate(gnd_coords):
            distance_3d = np.sqrt((tx_x-rx_x)**2 +(tx_y-rx_y)**2 +(tx_z-rx_z)**2)
            features.append({
                "TX": tx_id + 1,
                "RX": rx_id + 1,
                "3D_Distance": distance_3d,
                "TX_x": tx_x,
                "TX_y": tx_y,
                "TX_z": tx_z,
                "RX_x": rx_x,
                "RX_y": rx_y,
                "RX_z": rx_z
            })
    df = pd.DataFrame(features)
    if output_file is not None:
        df.to_csv(output_file, index=False)
    return df

def read_dxf_buildings(dxf_file, default_height=10):
    """
    Extract building geometries and geometric attributes from a DXF file.

    The function supports three DXF entity types:

    - LWPOLYLINE: interpreted as 2D building footprints.
    - POLYLINE: interpreted as 2D building footprints.
    - 3DFACE: interpreted as building faces from which a height is
      estimated from the Z coordinates.

    For 2D footprints, the building height cannot be obtained directly
    from the geometry and ``default_height`` is therefore assigned.

    Parameters
    ----------
    dxf_file : str or pathlib.Path
        Path to the DXF file containing building geometry.
    default_height : float, optional
        Height assigned to 2D building footprints when no Z information
        is available. Default is 10 m.

    Returns
    -------
    geopandas.GeoDataFrame
        Building geometries and attributes with the following columns:

        geometry
            Shapely Polygon representing the building footprint.

        height
            Building height in meters.

        num_faces
            Number of faces associated with the extracted building geometry.

        num_vertices
            Number of vertices defining the building geometry.

    Raises
    ------
    ValueError
        If no supported building geometry is found in the DXF file.
    """

    doc = ezdxf.readfile(dxf_file)
    msp = doc.modelspace()
    buildings = []
    entity_count = {}
    for entity in msp:
        etype = entity.dxftype()
        entity_count[etype] = entity_count.get(etype, 0) + 1
        # -------------------------
        # 2D footprint polylines
        # -------------------------
        if etype == "LWPOLYLINE":
            pts = [(p[0], p[1]) for p in entity.get_points()]
            if len(pts) >= 3:
                if pts[0] != pts[-1]:
                    pts.append(pts[0])
                poly = Polygon(pts)
                if poly.is_valid:
                    buildings.append({
                        "geometry": poly,
                        "height": default_height,
                        "num_faces": len(pts)-1,
                        "num_vertices": len(pts)-1
                    })
        # -------------------------
        # Old style polyline
        # -------------------------
        elif etype == "POLYLINE":
            pts = []
            for v in entity.vertices:
                pts.append((v.dxf.location.x,v.dxf.location.y))
            if len(pts) >= 3:
                if pts[0] != pts[-1]:
                    pts.append(pts[0])
                poly = Polygon(pts)
                if poly.is_valid:
                    buildings.append({
                        "geometry": poly,
                        "height": default_height,
                        "num_faces": len(pts)-1,
                        "num_vertices": len(pts)-1
                    })
        # -------------------------
        # 3D building faces
        # -------------------------
        elif etype == "3DFACE":
            pts = []
            for p in entity.wcs_vertices():
                pts.append((p.x,p.y))
            # remove duplicate points
            pts = list(dict.fromkeys(pts))
            if len(pts) >= 3:
                poly = Polygon(pts)
                if poly.is_valid:
                    # Estimate the vertical extent of the 3DFACE from its Z coordinates.
                    # If the face has no positive vertical extent, fall back to the
                    # user-defined default height.
                    z_values = [p.z for p in entity.wcs_vertices()]
                    height = max(z_values) - min(z_values)
                    if height <= 0:
                        height = default_height
                    buildings.append({
                        "geometry": poly,
                        "height": height,
                        "num_faces": 1,
                        "num_vertices": len(pts)
                    })
    print("DXF entity summary:")
    print(entity_count)
    print("Extracted buildings:", len(buildings))
    if len(buildings) == 0:
        raise ValueError("No building geometry found. ""Check DXF entity types.")
    # Assign the coordinate reference system used by the input geometry.
    gdf = gpd.GeoDataFrame(buildings,geometry="geometry",crs="EPSG:3857")
    return gdf




## Run Experiments
The following cells execute the experiments for the Baseline1 representation.

In [ ]:
PLOT_DEBUG = False
# Process each configured simulation scenario independently.
for idx, city in cities.items():
    # Retrieve input and output paths for the current scenario.
    name = city["name"]
    uav_file = city["uav_file"]
    gnd_file = city["gnd_file"]
    dxf_file = city["dxf_file"]

    output_file = city["output_file"]
    

    print(f"\nProcessing city {idx}: {name}")
    print("UAV:", uav_file)
    print("GND:", gnd_file)
    print("DXF:", dxf_file)

    # ---------------------------------------------------------
    # 1. Read UAV & GND coordinates
    # ---------------------------------------------------------
    uav_df, gnd_df, uav_coords, gnd_coords = read_tx_rx(uav_file,gnd_file)

    print("Loaded UAV:",uav_coords.shape,"Loaded GND:",gnd_coords.shape)
    # ---------------------------------------------------------
    # 2. Extract building and link-level geometric features
    # ---------------------------------------------------------

    # Convert DXF building geometry into a GeoDataFrame.
    buildings_gdf = read_dxf_buildings(dxf_file)
    # Extract building-environment features around each UAV.
    tx_features = extract_tx_features(buildings_gdf,uav_coords,section_size=100)   
    # Generate features for every UAV-RX combination.
    df_all = extract_all_features_Baseline2(output_file=None,uav_coords=uav_coords,gnd_coords=gnd_coords)   
    # Merge UAV-level building features with every RX associated
    # with the same UAV transmitter.
    df_all = df_all.merge(tx_features,on="TX",how="left")
    # Save the final feature dataset.
    df_all.to_csv(output_file, index=False)
    
    # ---------------------------------------------------------
    # 3. Optional visualization for debugging
    # ---------------------------------------------------------
    if PLOT_DEBUG:
        plot_dxf_with_first_tx_rx(dxf_file,uav_coords,gnd_coords)

    
